In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from scipy.spatial import distance
from scipy import spatial
import numpy as np


In [2]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

In [3]:
products = [{
        "title": "Smartphone X1",
        "short_description": "Flagship smartphone with AI features and 5G connectivity.",
        "price": 799.99,
        "category": "Electronics",
        "features": [
            "6.5-inch AMOLED display",
            "Triple camera system",
            "5G support",
            "Fast wireless charging"
        ]
    },
    {
        "title": "Laptop Pro 15",
        "short_description": "High-performance laptop for professionals and developers.",
        "price": 1299.99,
        "category": "Electronics",
        "features": [
            "15-inch Retina display",
            "16GB RAM",
            "512GB SSD",
            "Intel i7 processor"
        ]
    },
    {
        "title": "Wireless Headphones Z",
        "short_description": "Noise-cancelling headphones with immersive sound.",
        "price": 199.99,
        "category": "Electronics",
        "features": [
            "Active noise cancellation",
            "Bluetooth 5.0",
            "30-hour battery life",
            "Comfort fit design"
        ]
    },
    {
        "title": "Smartwatch Fit 2",
        "short_description": "Fitness-focused smartwatch with health tracking.",
        "price": 149.99,
        "category": "Wearables",
        "features": [
            "Heart rate monitoring",
            "Sleep tracking",
            "Water resistant",
            "GPS tracking"
        ]
    },
    {
        "title": "Gaming Console Ultra",
        "short_description": "Next-gen gaming console with ultra-fast performance.",
        "price": 499.99,
        "category": "Gaming",
        "features": [
            "4K gaming",
            "1TB storage",
            "Ray tracing support",
            "Wireless controller"
        ]
    },
    {
        "title": "Bluetooth Speaker Boom",
        "short_description": "Portable speaker with powerful bass and long battery.",
        "price": 89.99,
        "category": "Audio",
        "features": [
            "360-degree sound",
            "Waterproof",
            "12-hour battery",
            "Compact design"
        ]
    },
    {
        "title": "DSLR Camera ProShot",
        "short_description": "Professional DSLR camera for high-quality photography.",
        "price": 999.99,
        "category": "Photography",
        "features": [
            "24MP sensor",
            "4K video recording",
            "Interchangeable lenses",
            "Image stabilization"
        ]
    },
    {
        "title": "Tablet Air 10",
        "short_description": "Lightweight tablet for work and entertainment.",
        "price": 399.99,
        "category": "Electronics",
        "features": [
            "10-inch display",
            "128GB storage",
            "Stylus support",
            "Long battery life"
        ]
    },
    {
        "title": "Fitness Tracker Band",
        "short_description": "Affordable fitness tracker with essential features.",
        "price": 49.99,
        "category": "Wearables",
        "features": [
            "Step counter",
            "Calorie tracking",
            "Sleep monitoring",
            "Long battery life"
        ]
    },
    {
        "title": "Mechanical Keyboard RGB",
        "short_description": "Gaming keyboard with customizable RGB lighting.",
        "price": 129.99,
        "category": "Accessories",
        "features": [
            "Mechanical switches",
            "RGB lighting",
            "Anti-ghosting",
            "Ergonomic design"
        ]
    }
]

last_product = {
        "title": "Smartphone X1",
        "short_description": "Flagship smartphone with AI features and 5G connectivity.",
        "price": 799.99,
        "category": "Electronics",
        "features": [
            "6.5-inch AMOLED display",
            "Triple camera system",
            "5G support",
            "Fast wireless charging"]
        
    }

In [4]:
def create_product_text(product):
	return f"""
    Headline:{product['title']}
    Short_description:{product['short_description']}
    Price:{product['price']}
    Category:{product['category']}
    Features:{', '.join(product['features'])}
    """

In [7]:
product_texts=[create_product_text(product) for product in products]

last_product_text = create_product_text(last_product)
#print(article_texts)

In [5]:
def create_embeddings(texts):
    response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
    )
    response_dict = response.model_dump()
    
    return [data['embedding'] for data in response_dict['data']]



In [8]:
# Embed last_product_text and product_texts
last_product_embeddings = create_embeddings(last_product_text)[0] # [[]]

product_embeddings = create_embeddings(product_texts)



In [9]:
def find_n_closest(query_vector, embeddings, n=3):

    distances= []

    for i, embedding in enumerate(embeddings):
        distance=spatial.distance.cosine(query_vector,embedding)

        distances.append({
       "index":i,
        "distance":distance
        })

    distances=sorted(distances, key=lambda x:x["distance"]) 

    return distances[:n]

In [10]:
hits = find_n_closest(last_product_embeddings, product_embeddings) 

print(hits)

[{'index': 0, 'distance': np.float64(0.0)}, {'index': 4, 'distance': np.float64(0.42623399306395404)}, {'index': 7, 'distance': np.float64(0.44541110409175166)}]


In [11]:
for hit in hits:
     product= products[hit['index']]
     print(product['title'])

Smartphone X1
Gaming Console Ultra
Tablet Air 10


In [12]:
user_history = [{
        "title": "Smartphone X1",
        "short_description": "Flagship smartphone with AI features and 5G connectivity.",
        "price": 799.99,
        "category": "Electronics",
        "features": [
            "6.5-inch AMOLED display",
            "Triple camera system",
            "5G support",
            "Fast wireless charging"
        ]
    },
    {
        "title": "Laptop Pro 15",
        "short_description": "High-performance laptop for professionals and developers.",
        "price": 1299.99,
        "category": "Electronics",
        "features": [
            "15-inch Retina display",
            "16GB RAM",
            "512GB SSD",
            "Intel i7 processor"
        ]
    }]

In [13]:
# Prepare and embed the user_history, and calculate the mean embeddings
history_texts = [create_product_text(product) for product in user_history]
history_embeddings = create_embeddings(history_texts)
mean_history_embeddings = np.mean(history_embeddings, axis=0)

In [14]:
# Filter products to remove any in user_history
products_filtered = [product for product in products if product not in user_history]

In [15]:
# Combine product features and embed the resulting texts
product_texts = [create_product_text(product) for product in products_filtered]
product_embeddings = create_embeddings(product_texts)

In [16]:
hits = find_n_closest(mean_history_embeddings, product_embeddings)

for hit in hits:
  product = products_filtered[hit['index']]
  print(product['title'])

Tablet Air 10
Gaming Console Ultra
DSLR Camera ProShot
